# 04 - REST API Client Demonstration

This notebook demonstrates how to interact with the DERIM middleware
REST API using Python's `httpx` library. It covers device registration,
telemetry ingestion and retrieval, control commands, and forecasting.

## Prerequisites

Start the DERIM API server before running this notebook:

```bash
cd derim-middleware
uvicorn derim.main:app --reload --host 0.0.0.0 --port 8000
```

---

In [ ]:
import httpx
import json
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta, timezone

BASE_URL = 'http://localhost:8000/api/v1'
client = httpx.Client(base_url=BASE_URL, timeout=30.0)

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 5)

## 1. Health Check

In [ ]:
health = httpx.get('http://localhost:8000/health')
print(f'Status: {health.status_code}')
print(json.dumps(health.json(), indent=2))

## 2. Register Devices

In [ ]:
devices = [
    {
        'device_id': 'solar-inv-001',
        'device_type': 'solar_pv',
        'name': 'Rooftop Solar Inverter #1',
        'location': 'Building A - Roof',
        'protocol': 'modbus',
        'rated_power_kw': 5.0,
        'state': 'on',
    },
    {
        'device_id': 'bess-001',
        'device_type': 'battery',
        'name': 'Battery Storage Unit #1',
        'location': 'Building A - Basement',
        'protocol': 'mqtt',
        'rated_power_kw': 10.0,
        'state': 'on',
    },
    {
        'device_id': 'evse-001',
        'device_type': 'ev_charger',
        'name': 'EV Charger Station #1',
        'location': 'Parking Lot B',
        'protocol': 'ocpp',
        'rated_power_kw': 22.0,
        'state': 'standby',
    },
]

for device in devices:
    resp = client.post('/devices', json=device)
    print(f"Registered: {device['device_id']} -> {resp.status_code}")

## 3. List All Devices

In [ ]:
resp = client.get('/devices')
devices_list = resp.json()
print(f'Registered devices: {len(devices_list)}')
pd.DataFrame(devices_list)

## 4. Ingest Telemetry Data

In [ ]:
# Load sample data and ingest via API
solar_df = pd.read_csv('../data/sample_solar_data.csv', parse_dates=['timestamp'])

# Ingest first 96 records (24 hours)
records = []
for _, row in solar_df.head(96).iterrows():
    records.append({
        'timestamp': row['timestamp'].isoformat(),
        'device_id': 'solar-inv-001',
        'device_type': 'solar_pv',
        'power_kw': row['power_kw'],
        'energy_kwh': row['energy_kwh'],
        'voltage_v': row['voltage_v'],
        'current_a': row['current_a'],
        'frequency_hz': row['frequency_hz'],
        'state': row['state'],
    })

resp = client.post('/telemetry/solar-inv-001', json=records)
print(f'Ingestion response: {resp.status_code}')
print(json.dumps(resp.json(), indent=2))

## 5. Query Telemetry

In [ ]:
# Query the last 24 hours of data
resp = client.get(
    '/telemetry/solar-inv-001',
    params={
        'start': '2025-06-01T00:00:00',
        'end': '2025-06-02T00:00:00',
        'limit': 100,
    }
)

telemetry = resp.json()
print(f'Retrieved {len(telemetry)} records')

if telemetry:
    tdf = pd.DataFrame(telemetry)
    tdf['timestamp'] = pd.to_datetime(tdf['timestamp'])
    
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.plot(tdf['timestamp'], tdf['power_kw'], marker='o', markersize=2, linewidth=1)
    ax.set_title('Retrieved Telemetry: Solar Power Output', fontsize=13)
    ax.set_ylabel('Power (kW)')
    ax.set_xlabel('Time')
    plt.tight_layout()
    plt.show()

## 6. Send Control Command

In [ ]:
# Send a setpoint command
cmd = {
    'command': 'setpoint',
    'value': 3.5,
    'parameters': {'ramp_rate_kw_s': 0.5}
}

resp = client.post('/control/solar-inv-001', json=cmd)
print(f'Control response: {resp.status_code}')
print(json.dumps(resp.json(), indent=2))

## 7. Get Forecast

In [ ]:
resp = client.get('/forecast/solar-inv-001', params={'horizon_hours': 24})
forecast = resp.json()

print(f"Model: {forecast['model_name']}")
print(f"Horizon: {forecast['horizon_hours']} hours")
print(f"Points: {len(forecast['predictions'])}")

if forecast['predictions']:
    fdf = pd.DataFrame(forecast['predictions'])
    fdf['timestamp'] = pd.to_datetime(fdf['timestamp'])
    
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.plot(fdf['timestamp'], fdf['power_kw'], label='Forecast', linewidth=2, color='#E74C3C')
    if 'confidence_lower' in fdf.columns:
        ax.fill_between(
            fdf['timestamp'],
            fdf['confidence_lower'].fillna(0),
            fdf['confidence_upper'].fillna(0),
            alpha=0.2, color='#E74C3C', label='Confidence Interval'
        )
    ax.set_title('24-Hour Power Forecast', fontsize=13)
    ax.set_ylabel('Power (kW)')
    ax.set_xlabel('Time')
    ax.legend()
    plt.tight_layout()
    plt.show()

## 8. List Available Models

In [ ]:
resp = client.get('/forecast/models')
print('Available models:')
print(json.dumps(resp.json(), indent=2))

## Summary

This notebook demonstrated the complete API workflow:

1. **Device registration** via `POST /devices`.
2. **Telemetry ingestion** via `POST /telemetry/{device_id}`.
3. **Telemetry retrieval** via `GET /telemetry/{device_id}`.
4. **Control commands** via `POST /control/{device_id}`.
5. **Forecasting** via `GET /forecast/{device_id}`.

For the full API reference, visit `http://localhost:8000/docs` (Swagger UI)
or `http://localhost:8000/redoc` (ReDoc).